# 02 — Frozen v0.1.7 baseline on the complete test split

Runs the original global histogram-shifting baseline on **all 2,000 test images**, including PNG save/reload. This notebook is an audit of the frozen engineering baseline; it does **not** choose any v0.2.0 parameter. Per-image results are append-only/resumable.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, yaml
from rdhlab.io import read_gray
from rdhlab.codec import capacity_bits
from rdhlab.experiments import reversibility_trial
from rdhlab.pipeline import deterministic_seed

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
frac=float(config['baseline']['stress_fraction_of_global_capacity'])
manifest=pd.read_csv(config['dataset']['prepared_manifest'])
test=manifest[manifest.split=='test'].copy().reset_index(drop=True)
assert len(test)==config['dataset']['expected_splits']['test']==2000
out=Path('/workspace/results/baseline_v017'); out.mkdir(parents=True,exist_ok=True)
csv_path=out/'test_per_image.csv'
existing=pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
done=set(existing['source_id'].astype(str)) if len(existing) else set()
print('Test images:',len(test),'already complete:',len(done),'remaining:',len(test)-len(done))

Test images: 2000 already complete: 0 remaining: 2000


In [2]:
tmp=out/'_roundtrip.png'
for j,row in test.iterrows():
    sid=str(row.source_id)
    if sid in done: continue
    img=read_gray(row.path)
    cap=capacity_bits(img)
    n=max(1,min(cap,int(round(frac*cap))))
    rng=np.random.default_rng(deterministic_seed(seed,'baseline',sid))
    r=reversibility_trial(img,n,rng,tmp)
    r.update({
        'source_id':sid,'path':row.path,'capacity_bits':cap,
        'gross_bpp':n/img.size,'net_bpp':r['net_payload_bits']/img.size,
        'pixels':img.size,
    })
    pd.DataFrame([r]).to_csv(csv_path,mode='a',header=not csv_path.exists(),index=False)
    if (j+1)%100==0: print(f'{j+1}/2000')
if tmp.exists(): tmp.unlink()
print('Saved:',csv_path)

100/2000
200/2000
300/2000
400/2000
500/2000
600/2000
700/2000
800/2000
900/2000
1000/2000
1100/2000
1200/2000
1300/2000
1400/2000
1500/2000
1600/2000
1700/2000
1800/2000
1900/2000
2000/2000
Saved: /workspace/results/baseline_v017/test_per_image.csv


In [3]:
df=pd.read_csv(csv_path)
assert len(df)==2000
assert df['exact_image'].all() and df['exact_message'].all() and (df['ber']==0).all()
summary=df[['capacity_bits','sideinfo_bits','net_payload_bits','net_bpp','psnr','ssim','encode_ms','decode_ms']].describe(percentiles=[.05,.5,.95,.99]).T
display(summary)
summary.to_csv(out/'baseline_summary.csv')
print('Exact recovery after PNG I/O: PASS on all',len(df),'test images')

,count,mean,std,min,5%,50%,95%,99%,max
capacity_bits,2000.0,3053.788500,3753.309122,444.000000,749.000000,1947.000000,8787.550000,18625.150000,65536.000000
sideinfo_bits,2000.0,119.552000,184.116693,50.000000,50.000000,50.000000,554.000000,866.000000,1538.000000
net_payload_bits,2000.0,1407.341000,1888.600496,-1026.000000,161.900000,871.500000,4340.200000,9250.120000,32718.000000
net_bpp,2000.0,0.021474,0.028818,-0.015656,0.002470,0.013298,0.066226,0.141146,0.499237
psnr,2000.0,52.324781,3.548842,48.163397,48.415599,51.460515,59.332345,62.594408,70.613586
ssim,2000.0,0.999042,0.001871,0.965412,0.997083,0.999514,0.999837,0.999891,0.999962
encode_ms,2000.0,0.663892,0.312597,0.327190,0.396549,0.571834,1.245689,1.918325,3.847329
decode_ms,2000.0,0.491662,0.298128,0.163438,0.221451,0.396584,1.131850,1.655910,2.619184


Exact recovery after PNG I/O: PASS on all 2000 test images


The v0.1.7 baseline remains a **global codec**. v0.2.0 introduces a separate blockwise allocation experiment; do not overwrite these baseline files with v0.2.0 results.